# Nhóm 5 · 100 công ty · 10-K MD&A → Tone → CAR

Notebook tái kiểm dữ liệu thật FY2016–2025 đã lưu trong dự án. Không dùng dữ liệu mô phỏng. Nguồn và hạn chế ở `docs/REPORT.md`.

In [ ]:
from pathlib import Path
import sys
root=Path.cwd()
if root.name=='notebooks': root=root.parent
if str(root) not in sys.path:sys.path.insert(0,str(root))
import numpy as np,pandas as pd
from IPython.display import display
from tonecar.text import load_lm,score
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
print(root)

## 1. Đối chiếu danh sách và SEC manifest

Danh sách gốc là 100 ticker/CIK người dùng đưa. Mỗi ticker–FY chỉ có một accession; URL có thể mở trực tiếp tại SEC.

In [ ]:
source=pd.read_csv(root/'data/companies_100_input.csv')
manifest=pd.read_csv(root/'data/filings_manifest.csv')
panel=pd.read_csv(root/'outputs/real/same/car_panel.csv')
assert len(source)==100 and len(manifest)==1000 and len(panel)==1000
assert panel.ticker.nunique()==100
assert panel.accession.is_unique and not panel.duplicated(['ticker','fiscal_year']).any()
assert set(panel.accession)==set(manifest.accession)
display(manifest[['ticker','fiscal_year','accession','report_date','filing_date','source_url']].head())

## 2. Tái tính tone từ MD&A đã trích

Tone=(Positive−Negative)/TotalWords. Chạy cell này sẽ đọc 1.000 văn bản MD&A và từ điển LM, không cần mạng.

In [ ]:
lexicon=load_lm(root/'data/external/lm.csv')
for row in panel.itertuples():
    text=(root/'data/interim/mda'/(row.accession+'.txt')).read_text(encoding='utf-8')
    result=score(text,lexicon)
    assert result['word_count']==row.word_count
    assert np.isclose(result['tone'],row.tone,atol=1e-12)
print('Đúng',len(panel),'điểm tone');display(panel[['ticker','fiscal_year','positive_rate','negative_rate','uncertainty_rate','tone']].head(15))

## 3. Tái tính CAR từ AR từng phiên

Phiên t=0 căn ngày nộp SEC; mô hình thị trường được ước lượng ở [-120,-20]. Kiểm tra tổng AR bằng ba cửa sổ.

In [ ]:
ar=pd.read_csv(root/'outputs/real/same/event_ar.csv')
for lo,hi in [(-1,1),(-3,3),(-5,5)]:
    sums=ar[ar.relative_day.between(lo,hi)].groupby('event_id').abnormal_return.sum()
    assert np.allclose(panel.event_id.map(sums),panel[f'car_{lo}_{hi}'])
print('CAR khớp cả ba cửa sổ cho',len(panel),'event')

## 4. Hồi quy chính và ba cách căn ngày

In [ ]:
fit=smf.ols('Q("car_-1_1") ~ tone',panel).fit(cov_type='HC3')
reported=pd.read_csv(root/'outputs/real/same/regressions.csv')
base=reported[(reported.outcome=='car_-1_1')&(reported.specification=='tone')].iloc[0]
assert np.isclose(fit.params['tone'],base.coefficient) and np.isclose(fit.pvalues['tone'],base.p_value)
display(pd.DataFrame({'coefficient':fit.params,'HC3_SE':fit.bse,'p':fit.pvalues}))
for rule in ['same','next','acceptance']:
    frame=pd.read_csv(root/'outputs/real'/rule/'regressions.csv')
    display(frame[(frame.outcome=='car_-1_1')&(frame.specification=='tone')][['coefficient','p_value','ci_low','ci_high','n']].assign(alignment=rule))

## 5. Kiểm định độ nhạy và kiểm toán

Holm điều chỉnh toàn bộ 18 kiểm định định trước. Kiểm toán ranh giới là hỗ trợ AI và kiểm tra cấu trúc, chưa có nhãn toàn văn của con người.

In [ ]:
models=pd.read_csv(root/'outputs/limitations/sensitivity_models.csv')
assert len(models)==18 and np.allclose(multipletests(models.p_cluster,method='holm')[1],models.p_holm)
assert pd.read_csv(root/'data/exclusions.csv').empty
assert pd.read_csv(root/'data/extraction_audit.csv').accession.nunique()==1000
display(models[['alignment','specification','coefficient','p_cluster','p_holm','N']])
display(pd.read_csv(root/'outputs/real/same/cluster_se.csv'))

## Diễn giải

Xem cả hệ số, CI, mẫu và độ nhạy; không chọn kết quả chỉ vì p-value nhỏ. Mẫu 100 công ty có chủ đích và dữ liệu giá công khai không chứng minh quan hệ nhân quả hoặc chiến lược giao dịch. Xem `docs/REPORT.md` và `docs/LIMITATIONS_REMEDIATION.md`.